In [1]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [2]:
miso_flagged_ouliers = pd.read_parquet('outputs/lmp_outlier_flags.parquet')
print("Dataset shape:", miso_flagged_ouliers.shape)
miso_flagged_ouliers.head()

Dataset shape: (366313, 40)


,interval_start_utc,interval_end_utc,publish_time_utc,north_derated_outages_mw,north_forced_outages_mw,north_planned_outages_mw,north_unplanned_outages_mw,central_derated_outages_mw,central_forced_outages_mw,central_planned_outages_mw,...,north_solar,central_solar,south_solar,miso_solar,lmp_dam,energy_dam,congestion_dam,loss_dam,flag_selective,flag_above_132
0,2023-03-25 01:55:00+00:00,2023-03-25 02:00:00+00:00,2023-03-23 05:00:00+00:00,399.0,155.0,2895.0,285.0,1999.0,3006.0,18439.0,...,3.9,1.0,1.2,6.1,26.92,26.09,1.66,-0.83,False,False
1,2023-03-25 02:00:00+00:00,2023-03-25 02:05:00+00:00,2023-03-23 05:00:00+00:00,399.0,155.0,2895.0,285.0,1999.0,3006.0,18439.0,...,0.0,0.0,0.0,0.0,23.70,23.31,2.24,-1.85,False,False
2,2023-03-25 02:05:00+00:00,2023-03-25 02:10:00+00:00,2023-03-23 05:00:00+00:00,399.0,155.0,2895.0,285.0,1999.0,3006.0,18439.0,...,0.0,0.0,0.0,0.0,23.70,23.31,2.24,-1.85,False,False
3,2023-03-25 02:10:00+00:00,2023-03-25 02:15:00+00:00,2023-03-23 05:00:00+00:00,399.0,155.0,2895.0,285.0,1999.0,3006.0,18439.0,...,0.0,0.0,0.0,0.0,23.70,23.31,2.24,-1.85,False,False
4,2023-03-25 02:15:00+00:00,2023-03-25 02:20:00+00:00,2023-03-23 05:00:00+00:00,399.0,155.0,2895.0,285.0,1999.0,3006.0,18439.0,...,0.0,0.0,0.0,0.0,23.70,23.31,2.24,-1.85,False,False


In [ ]:
#features used for knn
knn_features = [
    "load_forecast",
    "north_wind",
    "central_wind",
    "south_wind",
    "north_solar",
    "central_solar",
    "south_solar",
    "lmp_dam",
    "energy_dam",
    "congestion_dam",
    "loss_dam"
]

print("Number of features:", len(knn_features))

Number of features: 11


In [ ]:
#Selective MAD method
miso_selective = miso_flagged_ouliers.copy()
#All prices above $132
miso_all_outliers = miso_flagged_ouliers.copy()

In [6]:
#mark values for knn impuations
#Selective method
miso_selective["lmp_imputed"] = miso_selective["lmp_rtm"].mask(miso_selective["flag_selective"])

#All-outlier method
miso_all_outliers["lmp_imputed"] = miso_all_outliers["lmp_rtm"].mask(miso_all_outliers["flag_above_132"])


In [10]:
#standardize data
scaler = StandardScaler()

X_scaled = scaler.fit_transform(miso_flagged_ouliers[knn_features])
print("Shape:", X_scaled.shape)
print("Means:", X_scaled.mean(axis=0).round(2))
print("Standard deviations:", X_scaled.std(axis=0).round(2))

Shape: (366313, 11)
Means: [-0.  0. -0. -0. -0.  0. -0.  0. -0.  0. -0.]
Standard deviations: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [11]:
#setting up knn
k = 5
knn = NearestNeighbors(n_neighbors=k)

In [12]:
#indentifying valid obs
#Selective
valid_selective = miso_selective["lmp_imputed"].notna()
#All-outlier
valid_all = miso_all_outliers["lmp_imputed"].notna()

print("Selective valid neighbors:", valid_selective.sum())
print("All-outlier valid neighbors:", valid_all.sum())

Selective valid neighbors: 358318
All-outlier valid neighbors: 352942


In [15]:
#selective knn model
knn_selective = NearestNeighbors(n_neighbors=5)
knn_selective.fit(X_scaled[valid_selective.to_numpy()])

,n_neighbors,5
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'minkowski'
,p,2
,metric_params,None
,n_jobs,None


In [16]:
#All-outlier KNN model
knn_all = NearestNeighbors(n_neighbors=5)
knn_all.fit(X_scaled[valid_all.to_numpy()])

,n_neighbors,5
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'minkowski'
,p,2
,metric_params,None
,n_jobs,None


In [ ]:
#varifiy models
print("Selective donor observations:", knn_selective.n_samples_fit_)
print("All-outlier donor observations:", knn_all.n_samples_fit_)

Selective donor observations: 358318
All-outlier donor observations: 352942


In [ ]:
#missing values
#Selective method
missing_selective = miso_selective["lmp_imputed"].isna()

#All-outlier method
missing_all = miso_all_outliers["lmp_imputed"].isna()

In [ ]:
#finding 5 nearest neighbors
distances_selective, indices_selective = (knn_selective.kneighbors( X_scaled[missing_selective.to_numpy()]))
distances_all, indices_all = (knn_all.kneighbors(X_scaled[missing_all.to_numpy()]))

print("Selective distances shape:", distances_selective.shape)
print("Selective indices shape:", indices_selective.shape)

print("All-outlier distances shape:", distances_all.shape)
print("All-outlier indices shape:", indices_all.shape)

Selective distances shape: (7995, 5)
Selective indices shape: (7995, 5)
All-outlier distances shape: (13371, 5)
All-outlier indices shape: (13371, 5)


In [23]:
#getting valid LMP prices
valid_prices_selective = miso_selective.loc[valid_selective, "lmp_imputed"].to_numpy()
valid_prices_all = miso_all_outliers.loc[valid_all, "lmp_imputed"].to_numpy()
#getting prices of the 5 nearest neighbors
neighbor_prices_selective = valid_prices_selective[indices_selective]
neighbor_prices_all = valid_prices_all[indices_all]


In [24]:
#inverse-distance weights
weights_selective = 1 / np.maximum(distances_selective, 1e-8)
weights_all = 1 / np.maximum(distances_all, 1e-8)

In [25]:
#estimated LMP
estimated_selective = ((neighbor_prices_selective * weights_selective).sum(axis=1)
                       / weights_selective.sum(axis=1)
)
estimated_all = ((neighbor_prices_all * weights_all).sum(axis=1)
                 / weights_all.sum(axis=1)
)

In [26]:
#filling in missing values
miso_selective.loc[missing_selective, 'lmp_imputed'] = estimated_selective
miso_all_outliers.loc[missing_all, 'lmp_imputed'] = estimated_all

In [27]:
#check to make sure nothing is missing
print("Selective missing values:", miso_selective["lmp_imputed"].isna().sum())
print("All-outlier missing values:", miso_all_outliers["lmp_imputed"].isna().sum())

Selective missing values: 0
All-outlier missing values: 0


In [28]:
#comparing summary statistics to imputation
comparison = pd.DataFrame({
    "Original": miso_flagged_ouliers["lmp_rtm"].describe(),
    "Selective KNN": miso_selective["lmp_imputed"].describe(),
    "All-Outlier KNN": miso_all_outliers["lmp_imputed"].describe()
})

comparison

,Original,Selective KNN,All-Outlier KNN
count,360711.000000,366313.000000,366313.000000
mean,34.408384,32.320938,28.959413
std,83.799619,65.424781,20.288822
min,-161.250000,-161.250000,-161.250000
25%,18.650000,18.640000,18.640000
50%,24.870000,24.860000,24.850000
75%,33.930000,33.870000,33.830000
max,10000.000000,10000.000000,132.000000


In [ ]:
# Create output folder
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Save selective KNN dataset
miso_selective.to_parquet(
    output_dir / "lmp_selective_knn.parquet",
    index=False
)

# Save all-outlier KNN dataset
miso_all_outliers.to_parquet(
    output_dir / "lmp_all_knn.parquet",
    index=False
)

print("Both KNN datasets saved successfully!")

Both KNN datasets saved successfully!
